# Recursividade e memória: o que a recursão gasta, onde, e como não estourar — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

A aula passada terminou com uma pergunta: `soma(v, 10000000)` funciona? Este tutorial retoma o
programa `memoria.c` visto em aula — duas somas recursivas de um vetor no heap e um Fibonacci com
tabela — e segue o mesmo caminho dos slides: o limite da pilha, o que pesa em cada registro, a
recursão de cauda, a tabela compartilhada no heap e o `malloc` em cada chamada. Depois de cada
bloco há um exercício com teste automático; no fim vêm as tarefas de modificação e o desafio.

## Objetivos

Ao final deste tutorial você será capaz de:

- Estimar a profundidade máxima de uma recursão a partir do tamanho da pilha e do tamanho de um
  registro, e reconhecer um estouro de pilha
- Reescrever uma recursão para que ela desça só $\log_2 n$ níveis
- Tirar da pilha os vetores que uma função recursiva usa, alocando-os uma vez no heap
- Escrever uma recursão de cauda e saber o que o compilador pode (e não precisa) fazer com ela
- Usar uma tabela no heap, compartilhada por todas as chamadas, para não recalcular resultados
- Alocar memória dentro de uma recursão sem vazamentos, liberando cada bloco na volta

In [ ]:
# Confira se o gcc está disponível e defina a função que testa os exercícios
!gcc --version | head -1

import subprocess

def testa(comando, esperado, tempo=20):
    '''Roda o comando e compara a saída com o esperado.'''
    try:
        r = subprocess.run(comando, shell=True, capture_output=True, text=True,
                           timeout=tempo)
    except subprocess.TimeoutExpired:
        print(f'Demorou mais de {tempo} s: a função está recalculando demais?')
        return
    saida = r.stdout.strip()
    if r.returncode != 0:
        print(f'O programa terminou com erro (status {r.returncode}; '
              '-11 ou 139 = segmentation fault).')
    if saida == esperado:
        print('OK')
    else:
        print(f'Verifique: esperava "{esperado}", veio "{saida}"')

## O programa da aula

Recompile o programa condutor **exatamente como saiu da aula**. Ele recebe o tamanho do vetor pela
linha de comando. Antes de cada execução, escreva a saída que você espera.

In [ ]:
%%writefile memoria.c
/* memoria.c --- quanto de memoria a recursao gasta, e onde */
#include <stdio.h>
#include <stdlib.h>
/* A: a soma da aula passada: a adicao fica para a volta */
long soma(const int *v, int n)
{
    if (n == 0)
        return 0;
    return soma(v, n - 1) + v[n - 1];
}

/* B: a mesma soma, levando o parcial na ida: nada fica para a volta */
long soma_acum(const int *v, int n, long parcial)
{
    if (n == 0)
        return parcial;
    return soma_acum(v, n - 1, parcial + v[n - 1]);
}

/* C: Fibonacci que anota cada resultado em uma tabela no heap */
long fib_memo(int n, long *memo)
{
    if (n < 2)
        return n;
    if (memo[n] == 0)
        memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo);
    return memo[n];
}

int main(int argc, char *argv[])
{
    int n = (argc > 1) ? atoi(argv[1]) : 1000;
    int *v = malloc(n * sizeof *v);         /* o vetor mora no heap */
    long *memo = calloc(91, sizeof *memo);  /* tabela zerada */
    if (v == NULL || memo == NULL) {
        free(v);
        free(memo);
        return 1;
    }
    for (int i = 0; i < n; i++)
        v[i] = 1;
    printf("C: fib(90) = %ld\n", fib_memo(90, memo));
    free(memo);

    printf("B: soma_acum(v, %d) = %ld\n", n, soma_acum(v, n, 0));
    printf("A: soma(v, %d) = %ld\n", n, soma(v, n));
    free(v);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra memoria.c -o memoria && ./memoria 100000

In [ ]:
!./memoria 1000000; echo "status: $?"

Em aula, no terminal, a linha `C:` apareceu antes do `Segmentation fault`. Aqui ela pode **não**
aparecer: quando a saída vai para um *pipe* (como no Jupyter), o `printf` acumula o texto em um
buffer e só o entrega no fim do programa — e um programa derrubado não chega ao fim. O status
`139` (= 128 + 11, o sinal `SIGSEGV`) é a única pista. Quando for depurar um programa que cai,
imprima em `stderr` ou chame `fflush(stdout)` depois de cada `printf`.

## 1. Até onde a pilha aguenta

Cada programa recebe, ao iniciar, uma pilha de tamanho **fixo** — em Linux e macOS, cerca de
8 MB. Cada chamada ativa ocupa um registro de ativação; quando a soma dos registros passa do
limite, o programa é derrubado com *segmentation fault*. Por isso:

$$\text{profundidade máxima} \approx \frac{\text{tamanho da pilha}}{\text{tamanho de um registro}}$$

A célula abaixo descobre, por busca binária, o maior `n` que `./memoria` aceita na sua máquina e
calcula quantos bytes cada registro ocupa.

In [ ]:
import resource

def cabe(programa, n):
    return subprocess.run([programa, str(n)], capture_output=True).returncode == 0

def maior_n(programa, lo=1, hi=20_000_000):
    '''Maior n para o qual o programa termina sem erro (busca binaria).'''
    while hi - lo > 1:
        meio = (lo + hi) // 2
        if cabe(programa, meio):
            lo = meio
        else:
            hi = meio
    return lo

pilha = resource.getrlimit(resource.RLIMIT_STACK)[0]
print(f'pilha: {pilha} bytes ({pilha // 1024} KB)')
limite = maior_n('./memoria')
print(f'maior n que passa: {limite}')
print(f'bytes por registro: ~{pilha / limite:.0f}')

O limite não depende de quantos elementos o vetor tem, e sim de **quantos níveis** a recursão
desce. `soma` e `soma_acum` descem um nível por elemento: $n$ registros para $n$ elementos.

Mas não precisa ser assim. Se a função somar a metade esquerda e a metade direita, cada nível
divide o problema por 2, e a profundidade cai para $\log_2 n$: para 10 milhões de elementos, uns
24 níveis.

In [ ]:
%%writefile ex1.c
#include <stdio.h>
#include <stdlib.h>

/* Exercicio 1: devolva a soma de v[ini..fim-1], somando separadamente
   a metade esquerda v[ini..meio-1] e a metade direita v[meio..fim-1].
   - nenhum elemento: soma 0;
   - um elemento: o proprio elemento.
   TODO: escreva soma_metades recursivamente.
   Ela deve funcionar para 10 milhoes de elementos SEM -O2. */
long soma_metades(const int *v, int ini, int fim)
{
    /* TODO: implemente aqui */
    return 0;
}

int main(void)
{
    int n = 10000000;
    int *v = malloc(n * sizeof *v);
    if (v == NULL)
        return 1;
    for (int i = 0; i < n; i++)
        v[i] = i % 10;
    printf("%ld %ld %ld\n", soma_metades(v, 0, 5), soma_metades(v, 0, n),
           soma_metades(v, 3, 3));
    free(v);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra ex1.c -o ex1
testa('./ex1', '10 45000000 0')

Quantos registros `soma_metades` empilha, no máximo, para $n = 10\,000\,000$? E quantas chamadas
ela faz **no total**? (Lembre da aula passada: chamadas no total $\neq$ chamadas ao mesmo tempo.)

## 2. O que pesa em cada registro

Parâmetros e variáveis locais — **inclusive vetores** — moram no registro de ativação. Numa
função recursiva, um vetor local é pago uma vez **por nível**. O programa abaixo tem um vetor de
4 KB em cada registro; descubra até onde ele aguenta.

In [ ]:
%%writefile gorda.c
/* gorda.c --- um vetor local em cada nivel da recursao */
#include <stdio.h>
#include <stdlib.h>

int desce(int n)
{
    char linha[4096];                   /* 4 KB em CADA registro */
    linha[0] = (char) n;
    if (n == 0)
        return linha[0];
    return desce(n - 1) + linha[0];
}

int main(int argc, char *argv[])
{
    int n = (argc > 1) ? atoi(argv[1]) : 100;
    printf("desce(%d) = %d\n", n, desce(n));
    return 0;
}

In [ ]:
!gcc -Wall -Wextra gorda.c -o gorda && ./gorda 100
limite = maior_n('./gorda', hi=200_000)
print(f'maior n que passa: {limite}; bytes por registro: ~{pilha / limite:.0f}')

O registro deixa de existir no `return`. Um ponteiro para uma local continua apontando para onde
ela estava — uma área que a próxima chamada reaproveita. O compilador avisa; leia o aviso e veja
o que sai.

In [ ]:
%%writefile rotulo.c
#include <stdio.h>

char *rotulo(int n)
{
    char texto[16];
    snprintf(texto, sizeof texto, "nivel %d", n);
    return texto;                 /* ERRADO: texto morre aqui */
}

int main(void)
{
    char *a = rotulo(1);
    char *b = rotulo(2);
    printf("%s | %s\n", a, b);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra rotulo.c -o rotulo && ./rotulo | od -c | head -3

(O `od -c` mostra os bytes um a um, porque o que sai nem sempre é texto válido.) A correção é
receber o espaço do chamador — `void rotulo(int n, char *dest, size_t tam)` — ou devolver um
bloco do `malloc` e documentar que quem chama deve dar `free`.

O exercício abaixo junta as duas ideias desta seção. A ordenação por intercalação divide o vetor
ao meio, ordena cada metade e intercala as duas, usando um vetor auxiliar. Ela desce só
$\log_2 n$ níveis — mas a versão abaixo declara o auxiliar como vetor **local** em cada chamada,
e o primeiro nível sozinho já pede 40 MB de pilha.

In [ ]:
%%writefile ex2.c
#include <stdio.h>
#include <stdlib.h>

/* intercala v[ini..meio-1] e v[meio..fim-1], ja' ordenados, usando aux */
static void intercala(int *v, int ini, int meio, int fim, int *aux)
{
    int i = ini, j = meio, k = 0;
    while (i < meio && j < fim)
        aux[k++] = (v[i] <= v[j]) ? v[i++] : v[j++];
    while (i < meio)
        aux[k++] = v[i++];
    while (j < fim)
        aux[k++] = v[j++];
    for (k = 0; k < fim - ini; k++)
        v[ini + k] = aux[k];
}

/* Exercicio 2: esta versao funciona para vetores pequenos, mas cai com
   10 milhoes de elementos: cada chamada declara um vetor local aux.
   TODO: aloque o auxiliar UMA VEZ, com malloc, em ordena; passe-o
   para baixo como parametro de ordena_rec; libere-o no fim.
   ordena devolve 1 se deu certo e 0 se faltou memoria. */
static void ordena_rec(int *v, int ini, int fim)
{
    if (fim - ini < 2)
        return;
    int aux[fim - ini];                 /* na pilha, em cada nivel */
    int meio = ini + (fim - ini) / 2;
    ordena_rec(v, ini, meio);
    ordena_rec(v, meio, fim);
    intercala(v, ini, meio, fim, aux);
}

int ordena(int *v, int n)
{
    ordena_rec(v, 0, n);
    return 1;
}

int main(void)
{
    int n = 10000000;
    int *v = malloc(n * sizeof *v);
    if (v == NULL)
        return 1;
    for (int i = 0; i < n; i++)
        v[i] = (int) ((i * 7919L) % n);   /* 0..n-1 embaralhados */
    if (!ordena(v, n))
        return 1;
    int ok = 1;
    for (int i = 0; i < n; i++)
        if (v[i] != i)
            ok = 0;
    printf("%s %d %d %d\n", ok ? "ordenado" : "ERRO", v[0], v[n / 2], v[n - 1]);
    free(v);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra ex2.c -o ex2
testa('./ex2', 'ordenado 0 5000000 9999999', tempo=60)

## 3. Recursão de cauda

Uma chamada é **de cauda** quando é a última coisa que a função faz: o valor dela é devolvido
direto, sem nenhuma conta depois. Em `soma`, a adição `+ v[n - 1]` fica pendente para a volta; em
`soma_acum`, o resultado parcial desce como parâmetro e nada fica pendente.

Se nada fica pendente, o registro de quem chamou não serve para mais nada, e o compilador **pode**
reaproveitá-lo — a recursão vira um laço. Compare a mesma execução sem e com `-O2`:

In [ ]:
!gcc -O2 -Wall -Wextra memoria.c -o memoria_O2
!./memoria 1000000 > /dev/null; echo "sem -O2: status $?"
!./memoria_O2 1000000; echo "com -O2: status $?"

Com `-O2`, a linha `B:` deve sair e o programa deve cair só em `soma`. Na sua máquina pode ser
diferente: o padrão C **não exige** essa otimização, e cada compilador decide o que fazer (alguns
conseguem transformar até a `soma`). Por isso a regra prática é: se a profundidade pode chegar a
dezenas de milhares, escreva o laço você mesmo.

In [ ]:
%%writefile ex3.c
#include <stdio.h>
#include <stdlib.h>

/* Exercicio 3: devolva o maior entre `maior` e os elementos v[0..n-1].
   TODO: escreva maximo_acum como RECURSAO DE CAUDA: a chamada recursiva
   e' a ultima coisa que a funcao faz, levando o maior ate' agora como
   parametro. Nada de conta depois da chamada. */
int maximo_acum(const int *v, int n, int maior)
{
    /* TODO: implemente aqui */
    return 0;
}

/* o maior de v[0..n-1], n >= 1 */
int maximo(const int *v, int n)
{
    return maximo_acum(v, n - 1, v[n - 1]);
}

int main(void)
{
    int a[] = {3, 9, 2, 7};
    int b[] = {-5};
    int c[] = {-8, -3, -12};
    int n = 1000000;
    int *g = malloc(n * sizeof *g);
    if (g == NULL)
        return 1;
    for (int i = 0; i < n; i++)
        g[i] = (i * 37) % n;
    printf("%d %d %d %d\n", maximo(a, 4), maximo(b, 1), maximo(c, 3),
           maximo(g, n));
    free(g);
    return 0;
}

In [ ]:
!gcc -O2 -Wall -Wextra ex3.c -o ex3
testa('./ex3', '9 -5 -3 999999')

Agora compile o mesmo `ex3.c` **sem** `-O2` e rode de novo. O que acontece? Por quê?

In [ ]:
!gcc -Wall -Wextra ex3.c -o ex3_O0 && ./ex3_O0; echo "status $?"

## 4. Uma tabela para todas as chamadas

A `fib` ingênua recalcula os mesmos valores do zero e faz $2F(n+1) - 1$ chamadas. A `fib_memo`
de `memoria.c` guarda cada resultado em uma tabela e consulta antes de recalcular
(**memoização**). A tabela é **uma só**, no heap, alocada pelo `main` antes da primeira chamada e
liberada depois da última; cada registro carrega só uma cópia do ponteiro.

O programa abaixo conta as chamadas das duas versões.

In [ ]:
%%writefile fib_compara.c
#include <stdio.h>
#include <stdlib.h>

long fib(int n, long *chamadas)
{
    (*chamadas)++;
    if (n < 2)
        return n;
    return fib(n - 1, chamadas) + fib(n - 2, chamadas);
}

long fib_memo(int n, long *memo, long *chamadas)
{
    (*chamadas)++;
    if (n < 2)
        return n;
    if (memo[n] == 0)
        memo[n] = fib_memo(n - 1, memo, chamadas)
                + fib_memo(n - 2, memo, chamadas);
    return memo[n];
}

int main(void)
{
    for (int n = 1; n <= 32; n++) {
        long k1 = 0, k2 = 0;
        long *memo = calloc(n + 1, sizeof *memo);
        if (memo == NULL)
            return 1;
        long a = fib(n, &k1);
        long b = fib_memo(n, memo, &k2);
        printf("%d %ld %ld %ld %ld\n", n, a, b, k1, k2);
        free(memo);
    }
    return 0;
}

In [ ]:
!gcc -Wall -Wextra fib_compara.c -o fib_compara && ./fib_compara | tail -3

In [ ]:
# Visualização: chamadas da fib ingênua x da fib com tabela
import matplotlib.pyplot as plt

linhas = subprocess.run(['./fib_compara'], capture_output=True, text=True).stdout.split('\n')
dados = [tuple(map(int, l.split())) for l in linhas if l.strip()]
ns       = [d[0] for d in dados]
ingenua  = [d[3] for d in dados]
memo     = [d[4] for d in dados]

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(ns, ingenua, 'o-', label='fib ingênua')
ax.semilogy(ns, memo, 's-', label='fib com tabela no heap')
ax.set_xlabel('n')
ax.set_ylabel('chamadas (escala log)')
ax.set_title('Memoização: memória por tempo')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.show()

A curva da ingênua é quase uma reta em escala log (crescimento exponencial); a da tabela é
$2n - 1$. O preço: $n + 1$ valores `long` no heap — para `fib(90)`, 728 bytes.

Repare no `calloc`: o zero significa "ainda não calculado". Com `malloc`, a tabela viria com lixo,
e o lixo seria lido como resultado pronto.

In [ ]:
%%writefile ex4.c
#include <stdio.h>
#include <stdlib.h>

/* Exercicio 4: C(n, k) = C(n-1, k-1) + C(n-1, k), com C(n, 0) = C(n, n) = 1.
   Sem tabela, C(60, 30) levaria seculos.
   TODO 1: em comb, consulte/preencha memo[n * nc + k] (zero = ainda nao
           calculado), como fib_memo faz com memo[n].
   TODO 2: em combinacoes, aloque com calloc uma tabela de (n+1) x (k+1)
           long em bloco unico, chame comb com nc = k + 1 e libere a
           tabela antes de devolver o resultado. */
static long comb(int n, int k, long *memo, int nc)
{
    /* TODO: implemente aqui */
    return 0;
}

long combinacoes(int n, int k)
{
    /* TODO: implemente aqui */
    return 0;
}

int main(void)
{
    printf("%ld %ld %ld %ld\n", combinacoes(5, 2), combinacoes(10, 0),
           combinacoes(30, 15), combinacoes(60, 30));
    return 0;
}

In [ ]:
!gcc -Wall -Wextra ex4.c -o ex4
testa('./ex4', '10 1 155117520 118264581564861424', tempo=10)

## 5. `malloc` em cada chamada

Quando uma função recursiva devolve memória do heap, cada nível recebe um bloco da chamada de
baixo, monta o seu e devolve. Quem **recebe** o bloco é dono dele: precisa usá-lo e depois dar
`free` — e esse `free` vem **depois** da chamada, ou seja, roda na volta.

`invertida.c` conta, com `aloca` e `libera`, quantos bytes estão no heap agora e no pico.

In [ ]:
%%writefile invertida.c
/* invertida.c --- malloc em cada chamada: quem libera, e quando? */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

static long vivos = 0, pico = 0;        /* bytes no heap agora e no maximo */

static char *aloca(size_t t)
{
    vivos += (long) t;
    if (vivos > pico)
        pico = vivos;
    return malloc(t);
}

static void libera(char *s)
{
    vivos -= (long) strlen(s) + 1;
    free(s);
}

/* devolve uma copia de s de tras para frente, alocada no heap */
char *invertida(const char *s)
{
    if (*s == '\0') {                   /* caso base: string vazia */
        char *r = aloca(1);
        r[0] = '\0';
        return r;
    }
    char *resto = invertida(s + 1);     /* o resto, ja' invertido */
    size_t k = strlen(resto);
    char *r = aloca(k + 2);
    strcpy(r, resto);                   /* o resto invertido...   */
    r[k] = *s;                          /* ...e a minha letra     */
    r[k + 1] = '\0';
    libera(resto);                      /* na volta: resto ja' nao serve */
    return r;
}

int main(int argc, char *argv[])
{
    int n = (argc > 1) ? atoi(argv[1]) : 10;
    char *s = malloc(n + 1);
    for (int i = 0; i < n; i++)
        s[i] = 'a' + i % 26;
    s[n] = '\0';

    char *r = invertida(s);
    printf("%.10s... (%zu letras)\n", r, strlen(r));
    printf("no heap agora: %ld bytes; pico: %ld bytes\n", vivos, pico);
    libera(r);
    free(s);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra invertida.c -o invertida
!for n in 10 1000 10000; do ./invertida $n; done

Agora a versão com vazamento: a mesma função, sem a linha `libera(resto);`.

In [ ]:
!sed 's|^    libera(resto);.*$|    /* libera(resto); */|' invertida.c > vaza.c
!gcc -Wall -Wextra vaza.c -o vaza
!for n in 10 1000 10000; do ./vaza $n; done

In [ ]:
# Visualização: pico do heap com e sem o free na volta
import re

def pico(programa, n):
    saida = subprocess.run([programa, str(n)], capture_output=True, text=True).stdout
    return int(re.search(r'pico: (\d+)', saida).group(1))

ns = list(range(500, 10001, 500))
com = [pico('./invertida', n) for n in ns]
sem = [pico('./vaza', n) for n in ns]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ns, [b / 1e6 for b in com], 'o-', label='com libera(resto)')
ax.plot(ns, [b / 1e6 for b in sem], 's-', label='sem libera(resto)')
ax.set_xlabel('letras na string (n)')
ax.set_ylabel('pico do heap (MB)')
ax.set_title('Um free esquecido dentro da recursão')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

Com o `free`, o pico é cerca de $2n$ bytes: só o bloco do nível de baixo e o que está sendo
montado existem juntos. Sem ele, nenhum bloco sai: $1 + 2 + \cdots + (n+1) = \frac{(n+1)(n+2)}{2}$
bytes, que crescem com o **quadrado** de $n$. E o resultado impresso é o mesmo nos dois casos —
nada avisa.

In [ ]:
%%writefile ex5.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

static long vivos = 0, pico = 0;        /* bytes no heap agora e no maximo */

static char *aloca(size_t t)
{
    vivos += (long) t;
    if (vivos > pico)
        pico = vivos;
    return malloc(t);
}

static void libera(char *s)
{
    vivos -= (long) strlen(s) + 1;
    free(s);
}

/* Exercicio 5: devolva, num bloco do heap, s repetida k vezes (k >= 0).
   Quem chama deve dar libera no resultado.
   TODO: escreva repete RECURSIVAMENTE, usando aloca e libera:
   - k == 0: a string vazia;
   - senao: s seguida de repete(s, k - 1).
   No fim do main, vivos deve voltar a 0. */
char *repete(const char *s, int k)
{
    /* TODO: implemente aqui */
    return NULL;
}

int main(void)
{
    char *a = repete("ab", 4);
    char *b = repete("xyz", 0);
    if (a == NULL || b == NULL)
        return 1;
    printf("[%s] [%s] ", a, b);
    libera(a);
    libera(b);
    printf("vivos=%ld pico=%ld\n", vivos, pico);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra ex5.c -o ex5
testa('./ex5', '[abababab] [] vivos=0 pico=16')

Por que o pico é 16 bytes? Escreva os tamanhos dos blocos de cada nível de `repete("ab", 4)` e
veja quais existem ao mesmo tempo.

## Tarefas

Trabalhe sobre uma cópia do programa da aula. A célula abaixo cria `memoria_tarefa.c` a partir de
`memoria.c`; edite esse arquivo (pelo Jupyter ou pelo seu editor) e recompile. **Antes de cada
execução, escreva a saída que você espera.**

**Tarefa 1.** Use `maior_n` (seção 1) para descobrir o maior `n` que o programa aceita sem e com
`-O2` na sua máquina. Com o tamanho da pilha, calcule o tamanho do registro de `soma_acum` e o de
`soma`.

**Tarefa 2.** Num terminal, rode `ulimit -s 65536` (pilha de 64 MB) e repita
`./memoria 1000000`: o que você espera? Depois volte ao limite normal, ponha um
`int rascunho[32]` em `soma_acum` (use-o, por exemplo `rascunho[0] = n;`, para o compilador não
apagá-lo) e preveja o novo limite antes de medir.

**Tarefa 3.** Troque `calloc` por `malloc` na tabela de `fib_memo`. O resultado muda? Sempre?
E se a tabela tiver 90 posições em vez de 91?

In [ ]:
!cp memoria.c memoria_tarefa.c
# edite memoria_tarefa.c e depois rode esta célula de novo (sem o cp, para não apagar sua edição)
!gcc -Wall -Wextra memoria_tarefa.c -o memoria_tarefa && ./memoria_tarefa 100000

## Desafio: pintar uma região

Uma grade de `l` × `c` caracteres, alocada no heap em bloco único (`g[i * c + j]`), tem células
livres (`.`) e paredes (`X`). Pintar a região que contém a célula `(i, j)` é marcar com `#` essa
célula e, a partir dela, todas as livres alcançáveis andando para cima, baixo, esquerda e direita.

- `pinta_rec`: se `(i, j)` está fora da grade ou não é `.`, devolve 0; senão marca com `#`,
  chama a si mesma para os quatro vizinhos e devolve 1 mais o que eles pintaram;
- `pinta_pilha`: o mesmo, **sem recursão**: uma pilha de posições (`i * c + j`) no heap, que
  começa pequena e dobra com `realloc` quando enche. Enquanto a pilha não estiver vazia,
  desempilhe uma posição; se ela ainda é `.`, pinte-a e empilhe os vizinhos livres.

Requisitos:

- as duas concordam na grade 5 × 9;
- numa grade 3000 × 3000, `pinta_pilha` pinta as 4 500 000 células e libera a própria pilha;
- `./desafio 1` roda a recursiva na grade grande e cai — mesmo com `-O2`. Explique por quê: ela
  é de cauda? Até quantos níveis ela pode descer?

In [ ]:
%%writefile desafio.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

/* TODO: versao recursiva; devolve quantas celulas pintou */
int pinta_rec(char *g, int l, int c, int i, int j)
{
    /* TODO: implemente aqui */
    return 0;
}

/* TODO: versao com uma pilha de posicoes no heap (malloc + realloc) */
int pinta_pilha(char *g, int l, int c, int i, int j)
{
    /* TODO: implemente aqui */
    return 0;
}

/* grade l x c toda livre, com uma parede vertical no meio */
static char *grade(int l, int c)
{
    char *g = malloc((size_t) l * c);
    if (g == NULL)
        return NULL;
    memset(g, '.', (size_t) l * c);
    for (int i = 0; i < l; i++)
        g[i * c + c / 2] = 'X';
    return g;
}

int main(int argc, char *argv[])
{
    int qual = (argc > 1) ? atoi(argv[1]) : 0;
    char *g = grade(5, 9);
    printf("rec 5x9: %d\n", pinta_rec(g, 5, 9, 0, 0));
    free(g);
    g = grade(5, 9);
    printf("pilha 5x9: %d\n", pinta_pilha(g, 5, 9, 0, 0));
    free(g);
    g = grade(3000, 3000);
    if (qual == 0)
        printf("pilha 3000x3000: %d\n", pinta_pilha(g, 3000, 3000, 0, 0));
    else
        printf("rec 3000x3000: %d\n", pinta_rec(g, 3000, 3000, 0, 0));
    free(g);
    return 0;
}

In [ ]:
!gcc -O2 -Wall -Wextra desafio.c -o desafio
testa('./desafio', 'rec 5x9: 20\npilha 5x9: 20\npilha 3000x3000: 4500000')
!./desafio 1; echo "recursiva na grade grande: status $?"

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. Para esta aula, os pontos de partida
são: a seção 3.7 de Bryant & O'Hallaron (a pilha de execução) e a 9.11 (erros comuns de memória);
as seções 6.3 e 36.3 de Kerrisk (o mapa de memória de um processo e o limite da pilha); a seção
1.2.1 de Abelson & Sussman (recursão e iteração, recursão de cauda); Michie (1968) e a seção 15.3
de Cormen et al. (memoização); e as seções 7.8.5 de Kernighan & Ritchie e 17.4 de King (`malloc`,
`free` e vazamentos).